## Поиск аномалий без учителя (Unsupervised Anomaly Detection)

В этом модуле мы решаем задачу детекции выбросов и подозрительных манипуляций с рейтингами:
1. **Отбор признаков**: используем числовые характеристики фильмов (рейтинг, сжатые голоса, индикатор аномальности, хронометраж, возраст).
2. **Масштабирование**: применяем `RobustScaler`, устойчивый к выбросам (на основе медианы и IQR).
3. **Модели поиска аномалий**: обучаем и сравниваем два разных подхода — **Isolation Forest** (изолирующие деревья) и **Local Outlier Factor (LOF)** (локальная плотность).
4. **Интерпретация**: находим конкретные подозрительные фильмы и визуализируем аномалии.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# 2. Загрузка очищенного датасета
df = pd.read_csv('data/movies_clean.csv')
print(f"Датасет успешно загружен: {len(df):,} строк, {df.shape[1]} колонок")

# 3. Отбор признаков для поиска аномалий
feature_cols = ['averageRating', 'log_votes', 'rating_to_log_votes', 'runtimeMinutes', 'movie_age']
X = df[feature_cols].copy()

print("\nПризнаки для анализа аномалий:")
X.head()

Датасет успешно загружен: 313,025 строк, 13 колонок

Признаки для анализа аномалий:


,averageRating,log_votes,rating_to_log_votes,runtimeMinutes,movie_age
0,5.3,2.378398,1.568791,45,132
1,5.3,2.788168,1.399093,100,129
2,6.0,3.041393,1.484637,70,120
3,5.0,1.623249,1.906033,90,119
4,5.0,1.929419,1.706823,120,118


### 2. Масштабирование признаков (RobustScaler)
Приводим признаки к сопоставимому масштабу. Почему именно **`RobustScaler`**:
- `StandardScaler` и `MinMaxScaler` сильно смещаются под влиянием выбросов.
- `RobustScaler` вычитает **медиану** и делит на **межквартильный размах (IQR = Q75 - Q25)**, поэтому сам скейлер устойчив к аномалиям, которые мы собираемся искать.

In [2]:
from sklearn.preprocessing import RobustScaler

# Инициализация и масштабирование матрицы признаков
scaler = RobustScaler()
X_scaled = scaler.fit_transform(X)

print(f"Форма X_scaled: {X_scaled.shape}")
print("Медианы после скейлинга (центрированы в 0):", np.round(np.median(X_scaled, axis=0), 2))
print("Межквартильный размах IQR (приведён к 1):", np.round(np.percentile(X_scaled, 75, axis=0) - np.percentile(X_scaled, 25, axis=0), 2))

Форма X_scaled: (313025, 5)
Медианы после скейлинга (центрированы в 0): [0. 0. 0. 0. 0.]
Межквартильный размах IQR (приведён к 1): [1. 1. 1. 1. 1.]


### 3. Детекция аномалий: Isolation Forest (Изолирующий лес)
Обучаем ансамбль изолирующих деревьев.
- **Принцип**: аномалии редки и сильно отличаются от большинства, поэтому в случайных деревьях они изолируются в неглубоких листьях (за малое число разбиений).
- **Параметры**:
  - `contamination=0.01`: ожидаем около 1% аномалий (3 130 фильмов).
  - `n_estimators=100`: 100 независимых деревьев для стабильного усреднения длины пути.
  - `random_state=42`: фиксация случайности для воспроизводимости.
  - `n_jobs=-1`: параллельное обучение на всех доступных ядрах процессора.

In [3]:
from sklearn.ensemble import IsolationForest

# 1. Инициализация модели Isolation Forest
iso_forest = IsolationForest(
    n_estimators=100,
    contamination=0.01,
    random_state=42,
    n_jobs=-1
)

# 2. Обучение модели на смасштабированных признаках
iso_forest.fit(X_scaled)

# 3. Получение меток (1: норма, -1: аномалия) и непрерывного скора
df['anomaly_iforest'] = iso_forest.predict(X_scaled)
df['score_iforest'] = iso_forest.decision_function(X_scaled)

n_anomalies = (df['anomaly_iforest'] == -1).sum()
print(f"Всего фильмов: {len(df):,}")
print(f"Обнаружено аномалий: {n_anomalies:,} ({n_anomalies / len(df):.2%})")
print("\nРаспределение скоров аномальности (score_iforest):")
print(df['score_iforest'].describe().round(3))

Всего фильмов: 313,025
Обнаружено аномалий: 3,131 (1.00%)

Распределение скоров аномальности (score_iforest):
count    313025.000
mean          0.145
std           0.050
min          -0.126
25%           0.115
50%           0.157
75%           0.185
max           0.224
Name: score_iforest, dtype: float64
